# 객체 YOLO TFLite 변환_Local

Colab에서 학습한 객체 모델 `best.pt`를 로컬 `Downloads`에 내려받은 뒤 TFLite integer/full integer 모델로 변환합니다.

입력:

```text
Downloads/best.pt
C:/Dataset/Train2YOLO_Outdoor_ObjectBox/data.yaml
```

출력:

```text
Downloads/OutdoorObject_TFLite/outdoor_object_yolo26n_integer_quant.tflite
```


In [ ]:
# 라즈베리파이 TensorFlow Lite 2.15.0 런타임과 맞추기 위한 변환 환경입니다.
# 이 셀을 실행한 뒤 반드시 커널을 재시작하고, 아래 셀들을 처음부터 다시 실행하세요.
# 최신 onnx는 TensorFlow 2.15가 요구하는 낮은 ml-dtypes와 충돌할 수 있어 함께 고정합니다.
%pip uninstall -y tensorflow tensorflow-cpu keras tf-keras ml-dtypes onnx onnxslim onnxruntime onnx2tf
%pip install "numpy<2" "protobuf<5" "ml-dtypes==0.2.0" "tensorflow==2.15.0" "keras==2.15.0" "tf-keras==2.15.0" "onnx==1.16.1" "onnxslim==0.1.34" ultralytics pyyaml


In [ ]:
import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

try:
    import tensorflow as tf
    print("tensorflow:", tf.__version__)
except Exception as e:
    raise RuntimeError(f"TensorFlow를 불러오지 못했습니다: {e}")

# Pi의 tflite_runtime/tensorflow-lite 런타임이 2.15.0이면 2.19.x에서 변환한 모델이
# TRANSPOSE_CONV 양자화 호환성 문제를 일으킬 수 있습니다.
if not tf.__version__.startswith("2.15."):
    raise RuntimeError(
        "현재 TensorFlow 버전이 2.15.x가 아닙니다. "
        "라즈베리파이 2.15.0 런타임용 TFLite는 TensorFlow 2.15.x 환경에서 다시 변환하세요."
    )

for package in ["ai_edge_litert", "tflite_runtime"]:
    try:
        module = __import__(package)
        version = getattr(module, "__version__", "installed")
        print(f"{package}: {version}")
    except Exception as e:
        print(f"{package}: not available ({e})")


In [ ]:
from pathlib import Path
import os
import random
import shutil
import yaml

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TF_NUM_INTRAOP_THREADS'] = '1'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

DOWNLOADS = Path.home() / 'Downloads'
MODEL_PT = DOWNLOADS / 'best.pt'
SOURCE_ROOT = Path('C:/Dataset/Train2YOLO_Outdoor_ObjectBox')
SOURCE_DATA_YAML = SOURCE_ROOT / 'data.yaml'
EXPORT_ROOT = DOWNLOADS / 'OutdoorObject_TFLite'
CALIB_ROOT = EXPORT_ROOT / 'calib_dataset'
CALIB_DATA_YAML = CALIB_ROOT / 'data.yaml'

IMGSZ = 512
CALIB_IMAGES_PER_SPLIT = 1000
CALIB_RANDOM_SEED = 42

if not MODEL_PT.exists():
    raise FileNotFoundError(MODEL_PT)
if not SOURCE_DATA_YAML.exists():
    raise FileNotFoundError(SOURCE_DATA_YAML)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

print('MODEL_PT:', MODEL_PT)
print('SOURCE_DATA_YAML:', SOURCE_DATA_YAML)
print('EXPORT_ROOT:', EXPORT_ROOT)


## calibration 데이터셋 생성

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
random.seed(CALIB_RANDOM_SEED)

source_data = yaml.safe_load(SOURCE_DATA_YAML.read_text(encoding='utf-8'))
names = source_data['names']

for split in ['train', 'val']:
    src_img_dir = SOURCE_ROOT / 'images' / split
    src_lbl_dir = SOURCE_ROOT / 'labels' / split
    dst_img_dir = CALIB_ROOT / 'images' / split
    dst_lbl_dir = CALIB_ROOT / 'labels' / split
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    images = sorted(p for p in src_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    selected = images if len(images) <= CALIB_IMAGES_PER_SPLIT else random.sample(images, CALIB_IMAGES_PER_SPLIT)
    for img in selected:
        shutil.copy2(img, dst_img_dir / img.name)
        src_lbl = src_lbl_dir / f'{img.stem}.txt'
        dst_lbl = dst_lbl_dir / f'{img.stem}.txt'
        if src_lbl.exists():
            shutil.copy2(src_lbl, dst_lbl)
        else:
            dst_lbl.touch(exist_ok=True)
    print(split, len(selected))

calib_data = {'path': str(CALIB_ROOT), 'train': 'images/train', 'val': 'images/val', 'names': names}
CALIB_DATA_YAML.write_text(yaml.safe_dump(calib_data, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(CALIB_DATA_YAML)


## TFLite 변환

In [ ]:
from ultralytics import YOLO

# export 직전에도 한 번 더 검사합니다. 이 셀만 단독 실행해도 2.19.x 변환을 막기 위함입니다.
import tensorflow as tf
if not tf.__version__.startswith('2.15.'):
    raise RuntimeError(
        f'현재 TensorFlow는 {tf.__version__}입니다. '
        '라즈베리파이 2.15.0 런타임용 변환은 TensorFlow 2.15.x에서만 실행하세요. '
        '설치 셀 실행 후 커널을 재시작해야 합니다.'
    )
print('변환 TensorFlow:', tf.__version__)



MODEL_STEM = 'outdoor_object_yolo26n'
FINAL_TFLITE_NAME = 'outdoor_object_yolo26n_integer_quant.tflite'
FINAL_PT_NAME = 'outdoor_object_yolo26n.pt'


def find_tflite_outputs(exported_path, roots):
    exported_path = Path(exported_path)
    search_roots = []
    if exported_path.is_file():
        search_roots.append(exported_path.parent)
    elif exported_path.is_dir():
        search_roots.append(exported_path)
    search_roots.extend(Path(r) for r in roots)

    found = []
    for root in search_roots:
        if root.exists():
            found.extend(root.rglob('*.tflite'))
    return sorted(set(found), key=lambda x: (x.stat().st_mtime, str(x)), reverse=True)


def pick_variant(candidates, suffix):
    matched = [
        p for p in candidates
        if p.name.endswith(suffix) and 'int16_act' not in p.name
    ]
    return matched[0] if matched else None


def backup_variant(candidates, suffix, target_name, required=False):
    src = pick_variant(candidates, suffix)
    if src is None:
        if required:
            print('TFLite 후보:')
            for p in candidates:
                print('-', p)
            raise FileNotFoundError(f'{suffix} 산출물을 찾지 못했습니다.')
        print(f'건너뜀: {suffix} 산출물이 없습니다.')
        return None

    dst = EXPORT_ROOT / target_name
    shutil.copy2(src, dst)
    print(f'백업: {src.name} -> {dst.name}')
    return dst


model = YOLO(str(MODEL_PT))

print('1) INT8/full integer 변환 시작')
exported_int8 = model.export(
    format='tflite',
    int8=True,
    data=str(CALIB_DATA_YAML),
    imgsz=IMGSZ,
    batch=1,
    device='cpu',
    nms=True,
)

int8_candidates = find_tflite_outputs(exported_int8, [MODEL_PT.parent, EXPORT_ROOT])
full_integer = backup_variant(
    int8_candidates,
    'full_integer_quant.tflite',
    f'{MODEL_STEM}_full_integer_quant.tflite',
    required=True,
)
backup_variant(int8_candidates, 'integer_quant.tflite', f'{MODEL_STEM}_integer_quant.tflite')
backup_variant(int8_candidates, 'int8.tflite', f'{MODEL_STEM}_int8.tflite')

# 기존 raspi5/config.py가 참조하던 파일명은 유지합니다.
final_tflite = EXPORT_ROOT / FINAL_TFLITE_NAME
shutil.copy2(full_integer, final_tflite)

print('\n2) float32 변환 시작')
exported_float32 = model.export(
    format='tflite',
    int8=False,
    imgsz=IMGSZ,
    batch=1,
    device='cpu',
    nms=True,
)

float32_candidates = find_tflite_outputs(exported_float32, [MODEL_PT.parent, EXPORT_ROOT])
backup_variant(float32_candidates, 'float32.tflite', f'{MODEL_STEM}_float32.tflite', required=True)

final_pt = EXPORT_ROOT / FINAL_PT_NAME
shutil.copy2(MODEL_PT, final_pt)

print('\n대표 TFLite:', final_tflite)
print('PT 백업:', final_pt)


## 결과 확인

In [ ]:
print('EXPORT_ROOT:', EXPORT_ROOT)
for p in sorted(EXPORT_ROOT.glob('*.tflite')):
    print(p.name, round(p.stat().st_size / (1024 * 1024), 2), 'MB')
